# Round 4 Manual Challenge: AETHER_CRYSTAL Options Optimization

This notebook values every listed contract using Black-Scholes where applicable and Monte Carlo simulation for path-dependent / conditional exotics. It then ranks trades by expected edge and evaluates portfolio-level risk across simulated paths.

**Important modeling assumptions:**
- Underlying starts at `S0 = 50`.
- GBM has zero risk-neutral drift and annualized volatility `sigma = 251%`.
- 252 trading days per year, 4 simulation steps per day.
- 2 weeks = 10 trading days = 40 steps.
- 3 weeks = 15 trading days = 60 steps.
- Contract size = 3000.
- Binary put payout is inferred as `20`, because `20 * P(S_T < 40)` is approximately 5.02, matching the quoted market.
- Knock-out put is modeled as strike 45 with down-and-out barrier 35. If the challenge uses a different barrier, update `KO_BARRIER` below.


In [ ]:
import math
import numpy as np
import pandas as pd

S0 = 50.0
SIGMA = 2.51
R = 0.0
TRADING_DAYS_PER_YEAR = 252
STEPS_PER_DAY = 4
STEPS_PER_YEAR = TRADING_DAYS_PER_YEAR * STEPS_PER_DAY
CONTRACT_SIZE = 3000

N_SIMS = 500_000
SEED = 20260427

def weeks_to_years(weeks: float) -> float:
    return (weeks * 5) / TRADING_DAYS_PER_YEAR

def steps_for_weeks(weeks: float) -> int:
    return int(round(weeks * 5 * STEPS_PER_DAY))

T2 = weeks_to_years(2)
T3 = weeks_to_years(3)
STEPS_2W = steps_for_weeks(2)
STEPS_3W = steps_for_weeks(3)

def norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

def bs_call(S, K, T, sigma=SIGMA, r=R):
    d1 = (math.log(S/K) + (r + 0.5*sigma*sigma)*T) / (sigma*math.sqrt(T))
    d2 = d1 - sigma*math.sqrt(T)
    return S*norm_cdf(d1) - K*math.exp(-r*T)*norm_cdf(d2)

def bs_put(S, K, T, sigma=SIGMA, r=R):
    d1 = (math.log(S/K) + (r + 0.5*sigma*sigma)*T) / (sigma*math.sqrt(T))
    d2 = d1 - sigma*math.sqrt(T)
    return K*math.exp(-r*T)*norm_cdf(-d2) - S*norm_cdf(-d1)

def digital_put(S, K, T, payout, sigma=SIGMA, r=R):
    d2 = (math.log(S/K) + (r - 0.5*sigma*sigma)*T) / (sigma*math.sqrt(T))
    return payout * math.exp(-r*T) * norm_cdf(-d2)


## Contract table

We use executable rows for every contract shown in the challenge. Empty orders are not automatically wrong: if a product is fairly priced or has negative expected edge after crossing the spread, the optimal EV/risk-aware order is often `0`.

In [ ]:
contracts = pd.DataFrame([
    dict(symbol='AC', kind='underlying', weeks=0, strike=np.nan, bid=49.975, ask=50.025, bid_size=200, ask_size=200),
    dict(symbol='AC_50_P', kind='put', weeks=3, strike=50, bid=12.00, ask=12.05, bid_size=50, ask_size=50),
    dict(symbol='AC_50_C', kind='call', weeks=3, strike=50, bid=12.00, ask=12.05, bid_size=50, ask_size=50),
    dict(symbol='AC_35_P', kind='put', weeks=3, strike=35, bid=4.33, ask=4.35, bid_size=50, ask_size=50),
    dict(symbol='AC_40_P', kind='put', weeks=3, strike=40, bid=6.50, ask=6.55, bid_size=50, ask_size=50),
    dict(symbol='AC_45_P', kind='put', weeks=3, strike=45, bid=9.05, ask=9.10, bid_size=50, ask_size=50),
    dict(symbol='AC_60_C', kind='call', weeks=3, strike=60, bid=8.80, ask=8.85, bid_size=50, ask_size=50),
    dict(symbol='AC_50_P_2', kind='put', weeks=2, strike=50, bid=9.70, ask=9.75, bid_size=50, ask_size=50),
    dict(symbol='AC_50_C_2', kind='call', weeks=2, strike=50, bid=9.70, ask=9.75, bid_size=50, ask_size=50),
    dict(symbol='AC_50_CO', kind='chooser', weeks=3, strike=50, bid=22.20, ask=22.30, bid_size=50, ask_size=50),
    dict(symbol='AC_40_BP', kind='binary_put', weeks=3, strike=40, bid=5.00, ask=5.10, bid_size=50, ask_size=50),
    dict(symbol='AC_45_KO', kind='ko_put', weeks=3, strike=45, bid=0.150, ask=0.175, bid_size=500, ask_size=500),
])

BINARY_PAYOUT = 20.0
KO_BARRIER = 35.0
contracts

## Monte Carlo path generation


In [ ]:
rng = np.random.default_rng(SEED)
dt = 1 / STEPS_PER_YEAR
z = rng.standard_normal((N_SIMS, STEPS_3W))
log_paths = math.log(S0) + np.cumsum((-0.5 * SIGMA**2) * dt + SIGMA * math.sqrt(dt) * z, axis=1)
paths = np.exp(log_paths)
S_2W = paths[:, STEPS_2W - 1]
S_3W = paths[:, STEPS_3W - 1]
path_min_3W = paths.min(axis=1)

pd.Series(S_3W).describe(percentiles=[.01,.05,.25,.5,.75,.95,.99])

## Fair values


In [ ]:
def mc_payoff(symbol):
    row = contracts.loc[contracts.symbol == symbol].iloc[0]
    K = row.strike
    if row.kind == 'underlying':
        return S_3W
    if row.kind == 'call':
        ST = S_2W if row.weeks == 2 else S_3W
        return np.maximum(ST - K, 0)
    if row.kind == 'put':
        ST = S_2W if row.weeks == 2 else S_3W
        return np.maximum(K - ST, 0)
    if row.kind == 'chooser':
        return np.where(S_2W >= K, np.maximum(S_3W - K, 0), np.maximum(K - S_3W, 0))
    if row.kind == 'binary_put':
        return BINARY_PAYOUT * (S_3W < K)
    if row.kind == 'ko_put':
        return np.where(path_min_3W > KO_BARRIER, np.maximum(K - S_3W, 0), 0)
    raise ValueError(row.kind)

def analytical_fair(row):
    if row.kind == 'underlying': return S0
    T = weeks_to_years(row.weeks)
    if row.kind == 'call': return bs_call(S0, row.strike, T)
    if row.kind == 'put': return bs_put(S0, row.strike, T)
    if row.kind == 'binary_put': return digital_put(S0, row.strike, T, BINARY_PAYOUT)
    return np.nan

rows = []
payoffs = {}
for _, row in contracts.iterrows():
    payoff = mc_payoff(row.symbol)
    payoffs[row.symbol] = payoff
    fair_mc = float(np.mean(payoff))
    fair_bs = analytical_fair(row)
    buy_ev = fair_mc - row.ask
    sell_ev = row.bid - fair_mc
    best_side = 'BUY' if buy_ev > sell_ev and buy_ev > 0 else ('SELL' if sell_ev > 0 else 'NO TRADE')
    best_ev = max(buy_ev, sell_ev, 0)
    rows.append({
        'symbol': row.symbol, 'kind': row.kind, 'bid': row.bid, 'ask': row.ask,
        'fair_mc': fair_mc, 'fair_bs_if_available': fair_bs,
        'buy_ev_per_unit': buy_ev, 'sell_ev_per_unit': sell_ev,
        'best_side_raw_ev': best_side, 'best_edge_per_unit': best_ev
    })

valuation = pd.DataFrame(rows).sort_values('best_edge_per_unit', ascending=False)
valuation

## Candidate order set

This sets nonzero orders only where the expected edge is positive enough to justify crossing the spread. You can force orders for every product, but that usually lowers expected PnL and increases risk.

In [ ]:
# Positive values = buy volume. Negative values = sell volume.
# This is the risk-aware recommendation under the assumptions above.
orders = {
    'AC': 0,
    'AC_50_P': 0,
    'AC_50_C': 0,
    'AC_35_P': 0,
    'AC_40_P': 0,
    'AC_45_P': 0,
    'AC_60_C': 0,
    'AC_50_P_2': 50,
    'AC_50_C_2': 50,
    'AC_50_CO': -50,
    'AC_40_BP': 0,
    'AC_45_KO': 500,
}

def position_pnl(symbol, qty):
    if qty == 0:
        return np.zeros(N_SIMS)
    row = contracts.loc[contracts.symbol == symbol].iloc[0]
    payoff = payoffs[symbol]
    if qty > 0:
        return qty * CONTRACT_SIZE * (payoff - row.ask)
    else:
        return (-qty) * CONTRACT_SIZE * (row.bid - payoff)

portfolio_pnl = sum(position_pnl(sym, qty) for sym, qty in orders.items())

risk_summary = pd.Series(portfolio_pnl).describe(percentiles=[.001,.005,.01,.05,.25,.5,.75,.95,.99,.995,.999])
risk_summary

In [ ]:
order_rows = []
for sym, qty in orders.items():
    pnl = position_pnl(sym, qty)
    order_rows.append({
        'symbol': sym,
        'qty': qty,
        'side': 'BUY' if qty > 0 else ('SELL' if qty < 0 else 'NO TRADE'),
        'volume': abs(qty),
        'expected_pnl': pnl.mean(),
        'p05_pnl': np.quantile(pnl, .05),
        'p01_pnl': np.quantile(pnl, .01),
        'worst_sim_pnl': pnl.min(),
    })

order_summary = pd.DataFrame(order_rows)
order_summary

## Alternative: all-positive-edge aggressive portfolio

This version takes every strictly positive raw EV signal. Compare it against the risk-aware set above. In practice, small positive edges on very volatile short gamma / binary products may not be worth the tail risk unless the objective is pure expected value.

In [ ]:
aggressive_orders = {}
for _, row in contracts.iterrows():
    v = valuation.loc[valuation.symbol == row.symbol].iloc[0]
    if v.buy_ev_per_unit > 0 and v.buy_ev_per_unit >= v.sell_ev_per_unit:
        aggressive_orders[row.symbol] = int(row.ask_size)
    elif v.sell_ev_per_unit > 0:
        aggressive_orders[row.symbol] = -int(row.bid_size)
    else:
        aggressive_orders[row.symbol] = 0

aggressive_pnl = sum(position_pnl(sym, qty) for sym, qty in aggressive_orders.items())
pd.DataFrame({'symbol': list(aggressive_orders), 'qty': list(aggressive_orders.values())})

In [ ]:
comparison = pd.DataFrame([
    dict(portfolio='risk_aware', mean=portfolio_pnl.mean(), std=portfolio_pnl.std(), p05=np.quantile(portfolio_pnl,.05), p01=np.quantile(portfolio_pnl,.01), min=portfolio_pnl.min(), sharpe_like=portfolio_pnl.mean()/portfolio_pnl.std()),
    dict(portfolio='aggressive_all_positive_ev', mean=aggressive_pnl.mean(), std=aggressive_pnl.std(), p05=np.quantile(aggressive_pnl,.05), p01=np.quantile(aggressive_pnl,.01), min=aggressive_pnl.min(), sharpe_like=aggressive_pnl.mean()/aggressive_pnl.std()),
])
comparison